# READ Bronze data - create data farame

In [1]:
df_order_raw = spark.read.format('parquet').load('abfss://292fd850-c68a-4bad-b4f3-7cabce198e06@onelake.dfs.fabric.microsoft.com/ce0794f0-6810-4e3e-84f0-12eac5f86b1f/Files/Bronze Data ( Raw Data )/Orders Data.Parquet')
df_returns_raw = spark.read.format('parquet').load('abfss://292fd850-c68a-4bad-b4f3-7cabce198e06@onelake.dfs.fabric.microsoft.com/ce0794f0-6810-4e3e-84f0-12eac5f86b1f/Files/Bronze Data ( Raw Data )/Returns Data.Parquet')
df_inventory_raw = spark.read.format('parquet').load('abfss://292fd850-c68a-4bad-b4f3-7cabce198e06@onelake.dfs.fabric.microsoft.com/ce0794f0-6810-4e3e-84f0-12eac5f86b1f/Files/Bronze Data ( Raw Data )/Inventory Data.Parquet')

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 3, Finished, Available, Finished, False)

In [3]:
display(df_order_raw.limit(5))

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 03128e6d-b4ff-40c7-b12c-f1205a8050af)

In [4]:
display(df_returns_raw.limit(5))

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d9781a5-2869-4d4f-8463-f135b315561d)

In [5]:
display(df_inventory_raw.limit(5))

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5c525a42-47d3-4aec-ac24-4cdb4369d6c7)

In [6]:
#### Making first row has header for data frame df_returns_raw

# Collect the first row (which contains header values)
first_row = df_returns_raw.first()

# Use that row's values as column names
new_columns = [str(c) for c in first_row]

# Drop the first row from the DataFrame
df_without_header = df_returns_raw.rdd.zipWithIndex() \
    .filter(lambda x: x[1] > 0) \
    .map(lambda x: x[0]) \
    .toDF()

# Apply the new column names
df_returns_raw = df_without_header.toDF(*new_columns)

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 8, Finished, Available, Finished, False)

In [7]:
display(df_returns_raw.limit(5))

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2bceb8da-b230-46ab-8928-f427a2b77bdf)

### create bronze delta tables 

In [9]:
df_order_raw.write.mode("overwrite").format("delta").saveAsTable('Retail_Analytics.Bronze_Orders')
df_returns_raw.write.mode("overwrite").format("delta").saveAsTable('Retail_Analytics.Bronze_Returns')
df_inventory_raw.write.mode("overwrite").format("delta").saveAsTable('Retail_Analytics.Bronze_Inventory')

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 11, Finished, Available, Finished, False)

In [10]:
%%sql 
select * from Retail_Analytics.Bronze_Orders
limit 10;

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 12, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 12 fields>

### clean the data - silver layer preparation 

In [11]:
df_order_bronze = spark.read.format('delta').load('abfss://292fd850-c68a-4bad-b4f3-7cabce198e06@onelake.dfs.fabric.microsoft.com/ce0794f0-6810-4e3e-84f0-12eac5f86b1f/Tables/Retail_Analytics/bronze_orders')
df_inventory_bronze = spark.read.format('delta').load('abfss://292fd850-c68a-4bad-b4f3-7cabce198e06@onelake.dfs.fabric.microsoft.com/ce0794f0-6810-4e3e-84f0-12eac5f86b1f/Tables/Retail_Analytics/bronze_inventory')
df_returns_bronze = spark.read.format('delta').load('abfss://292fd850-c68a-4bad-b4f3-7cabce198e06@onelake.dfs.fabric.microsoft.com/ce0794f0-6810-4e3e-84f0-12eac5f86b1f/Tables/Retail_Analytics/bronze_returns')


StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 13, Finished, Available, Finished, False)

In [12]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 14, Finished, Available, Finished, False)

### CLeaning Orders Data

In [13]:
df_orders = (
    df_order_bronze

    # 2. Clean column names
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("cust_id", "CustomerID")
    .withColumnRenamed("Product_Name", "ProductName")
    .withColumnRenamed("Qty", "Quantity")
    .withColumnRenamed("Order_Date", "OrderDate")
    .withColumnRenamed("Order_Amount$", "OrderAmount")
    .withColumnRenamed("Delivery_Status", "DeliveryStatus")
    .withColumnRenamed("Payment_Mode", "PaymentMode")
    .withColumnRenamed("Ship_Address", "ShipAddress")
    .withColumnRenamed("Promo_Code", "PromoCode")
    .withColumnRenamed("Feedback_Score", "FeedbackScore")

    # 3. Normalize Quantity: convert words like 'one', 'Two' to integer
    .withColumn("Quantity", 
        when(lower(col("Quantity")) == "one", 1)
        .when(lower(col("Quantity")) == "two", 2)
        .when(lower(col("Quantity")) == "three", 3)
        .otherwise(col("Quantity").cast(IntegerType()))
    )

    # 4. Standardize date format using multiple patterns
    .withColumn("OrderDate", to_date(
        coalesce(
            to_date(col("OrderDate"), "yyyy/MM/dd"),
            to_date(col("OrderDate"), "dd-MM-yyyy"),
            to_date(col("OrderDate"), "MM-dd-yyyy"),
            to_date(col("OrderDate"), "yyyy.MM.dd"),
            to_date(col("OrderDate"), "dd/MM/yyyy"),
            to_date(col("OrderDate"), "dd.MM.yyyy"),
            to_date(col("OrderDate"), "MMMM dd yyyy")
        )
    ))

    # 5. Clean and convert OrderAmount
    .withColumn("OrderAmount", regexp_replace(col("OrderAmount"), "[$₹Rs. USD, INR]", ""))
    .withColumn("OrderAmount", col("OrderAmount").cast(DoubleType()))

    # 6. Standardize PaymentMode ! 
    .withColumn("PaymentMode", lower(regexp_replace(col("PaymentMode"), "[^a-zA-Z]", "")))

    # 7. Standardize DeliveryStatus
    .withColumn("DeliveryStatus", lower(regexp_replace(col("DeliveryStatus"), "[^a-zA-Z ]", "")))

    # 8. Validate email using simple regex pattern
    .withColumn("Email", when(col("Email").rlike("^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"), col("Email")).otherwise(None))

    # 9. Clean address: remove special characters like #, !, $, @ etc.
    .withColumn("ShipAddress", regexp_replace(col("ShipAddress"), r"[#@!$]", ""))

    # 10. FeedbackScore: convert to float, handle NaN/bad values
    .withColumn("FeedbackScore", col("FeedbackScore").cast(DoubleType()))

    # 11. Fill nulls where possible
    .fillna({"Quantity": 0, "OrderAmount": 0.0, "DeliveryStatus": "unknown", "PaymentMode": "unknown"})

    # 12. Drop rows with no CustomerID or ProductName
    .na.drop(subset=["CustomerID", "ProductName"])

    # 13. Remove duplicates by OrderID
    .dropDuplicates(["OrderID"])
)

display(df_orders.limit(6))

# Save cleaned Orders data
df_orders.write.mode("overwrite").format("delta").saveAsTable("Retail_Analytics.silver_orders")


StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f513dfe5-6588-4106-b316-ec50fdbe5bf8)

### Cleaning Inventory Data

In [14]:
df_inventory = (
    df_inventory_bronze
    .withColumnRenamed("productName", "ProductName")
    .withColumnRenamed("cost_price", "CostPrice")
    .withColumnRenamed("last_stocked", "LastStocked")
    
    # 2. Clean stock column: convert to integer
    .withColumn("Stock", 
        when(col("stock").rlike("^[0-9]+$"), col("stock").cast(IntegerType()))  # Numeric values
        .when(col("stock").isNull() | (col("stock") == ""), lit(None))  # Null or blank
        .otherwise(
            when(col("stock").rlike(".*twenty five.*"), lit(25))
            .when(col("stock").rlike(".*twenty.*"), lit(20))
            .when(col("stock").rlike(".*eighteen.*"), lit(18))
            .when(col("stock").rlike(".*fifteen.*"), lit(15))
            .when(col("stock").rlike(".*twelve.*"), lit(12))
            .otherwise(lit(None))
        ).cast(IntegerType())
    )
    
    # 3. Clean LastStocked: normalize multiple date formats to yyyy-MM-dd
    .withColumn("LastStocked", to_date(
        regexp_replace("LastStocked", "[./]", "-"), "yyyy-MM-dd"
    ))
    
    # 4. Clean CostPrice: extract numeric value and convert to float
    .withColumn("CostPrice", 
        regexp_extract(col("CostPrice"), r"(\d+\.?\d*)", 1).cast(DoubleType())
    )

    # 5. Clean Warehouse: remove special characters, trim, capitalize first letter
    .withColumn("Warehouse", 
        initcap(trim(regexp_replace(col("warehouse"), r"[^a-zA-Z0-9\s]", " ")))
    )
    
    # 6. Standardize Available: convert to boolean
    .withColumn("Available", 
        when(lower(col("available")).isin("yes", "y", "true"), lit(True))
        .when(lower(col("available")).isin("no", "n", "false"), lit(False))
        .otherwise(None)
    )
    
)

# Display the cleaned data
display(df_inventory)

# Optional: Save to Silver Layer
df_inventory.write.mode("overwrite").format("delta").saveAsTable("Retail_Analytics.silver_inventory")

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b2e233e9-8a79-42d9-9f78-880853ffcb6a)

### Cleaning Returns Data

In [15]:

df_returns = (
    df_returns_bronze
    # 2.1 Standardize column names (if needed)
    .withColumnRenamed("Return_ID", "ReturnID")
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("Customer_ID", "CustomerID")
    .withColumnRenamed("Return_Reason", "ReturnReason")
    .withColumnRenamed("Return_Date", "ReturnDate")
    .withColumnRenamed("Refund_Status", "RefundStatus")
    .withColumnRenamed("Pickup_Address", "PickupAddress")
    .withColumnRenamed("Return_Amount", "ReturnAmount")
    
    # 2.2 Clean ReturnDate → standardize date formats
    .withColumn("ReturnDate", to_date(
        regexp_replace("ReturnDate", r"[./]", "-"), "dd-MM-yyyy"
    ))
    
    # 2.3 Clean RefundStatus → lowercase, remove special characters
    .withColumn("RefundStatus", lower(regexp_replace(col("RefundStatus"), r"[^a-zA-Z]", "")))
    
    # 2.4 Clean ReturnAmount → extract numeric part regardless of currency  1$ - 1
    .withColumn("ReturnAmount", 
        regexp_extract(col("ReturnAmount"), r"(\d+\.?\d*)", 1).cast(DoubleType())
    )
    
    # 2.5 Clean PickupAddress → remove special characters
    .withColumn("PickupAddress", initcap(trim(regexp_replace(col("PickupAddress"), r"[^a-zA-Z0-9\s]", " "))))
    
    # 2.6 Clean Product → remove extra symbols and spaces
    .withColumn("Product", initcap(trim(regexp_replace(col("Product"), r"[^a-zA-Z0-9\s]", ""))))
    
    # 2.7 Clean CustomerID → trim, fix wrong prefixes
    .withColumn("CustomerID", trim(upper(col("CustomerID"))))
    
)

# Step 3: Show cleaned Silver data
display(df_returns)

# Step 4: Save to Silver Delta Table
df_returns.write.mode("overwrite").format("delta").saveAsTable("Retail_Analytics.silver_returns")

StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a0ba8fab-4e48-4b97-8c31-3d144585105a)

### gold aggragte 

In [16]:
from pyspark.sql.functions import *

# STEP 1: Load cleaned Silver tables with aliases
orders = spark.table("Retail_Analytics.silver_orders").alias("o")
returns = spark.table("Retail_Analytics.silver_returns").alias("r")
inventory = spark.table("Retail_Analytics.silver_inventory").alias("i")

# STEP 2: Join Orders with Returns (LEFT)
order_return = orders.join(
    returns,
    on=col("o.OrderID") == col("r.OrderID"),
    how="left"
)

# STEP 3: Join with Inventory (LEFT)
enriched = order_return.join(
    inventory,
    on=col("o.ProductName") == col("i.ProductName"),
    how="left"
)



# STEP 5: Select explicit columns to avoid ambiguity
df_enriched = enriched.select(
    col("o.ProductName").alias("ProductName"),
    col("o.OrderID").alias("OrderID"),
    col("o.CustomerID").alias("CustomerID"),
    col("o.OrderAmount").alias("OrderAmount"),
    col("r.ReturnID").alias("ReturnID"),
    col("i.Stock").alias("Stock"),
    col("i.CostPrice").alias("CostPrice")
)

# STEP 6: Aggregate KPIs by Product and Month
df_kpi = (
    df_enriched.groupBy("ProductName")
    .agg(
        count("OrderID").alias("Total_Orders"),
        countDistinct("CustomerID").alias("Unique_Customers"),
        count("ReturnID").alias("Total_Returns"),
        round((count("ReturnID") / count("OrderID")) * 100, 2).alias("Return_Rate_%"),
        round(sum("OrderAmount"), 2).alias("Total_Revenue"),
        round(avg("OrderAmount"), 2).alias("Avg_Order_Value"),
        sum("Stock").alias("Total_Stock"),
        round(avg("CostPrice"), 2).alias("Avg_Cost"),
        round(sum("OrderAmount") - (sum("Stock") * avg("CostPrice")), 2).alias("Net_Profit")
    )
)

# STEP 7: Display results
display(df_kpi)

# (Optional) STEP 8: Save to Gold Delta table
df_kpi.write.mode("overwrite").format("delta").saveAsTable("Retail_Analytics.gold_product_month_kpis")


StatementMeta(, 35eb9bff-db84-4903-9e5a-17a97f42c3ff, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 25392a37-aace-485d-97cd-3c011da5cd4f)